# 11_build_74cls_5fold_dataset

원본 Colab 셀에서 분리. (`#@title [새 마스킹 기준] 조합단위 COCO 변환 + 5-fold 병합 + zip 백업`)


In [ ]:
#@title [매 세션 1] rfdetr 설치
#@markdown 런타임이 끊기면 설치된 패키지가 전부 사라지므로, 매 세션 rfdetr을 다시 설치해야 함
!pip install -q "rfdetr[train,loggers]"                  # RF-DETR 학습(train)+로깅(loggers) 의존성

In [ ]:
#@title [매 세션 2] 드라이브 마운트 + 경로 자동 탐색
#@markdown 세션마다 드라이브 연결이 끊기므로 재마운트 필요. 데이터·zip이 전부 드라이브에 있어 경로부터 다시 잡아야 함
from google.colab import drive                           # 코랩↔드라이브 연결 도구
drive.mount('/content/drive')                             # 드라이브 마운트

import os, glob                                          # 경로·탐색 도구
CANDS = [                                                # 흔한 후보 경로 먼저
    '/content/drive/MyDrive/1팀 공유 문서/ai12-level1-project/sprint_ai_project1_data',
    '/content/drive/MyDrive/ai12-level1-project/sprint_ai_project1_data',
]
DATA_ROOT = next((c for c in CANDS if os.path.exists(c)), None)   # 존재하는 첫 경로 채택
if DATA_ROOT is None:                                    # 후보에 없으면 전체 재귀 검색
    hits = glob.glob('/content/drive/MyDrive/**/sprint_ai_project1_data', recursive=True)
    DATA_ROOT = hits[0] if hits else None
assert DATA_ROOT, "sprint_ai_project1_data를 못 찾음"     # 못 찾으면 중단
PROJ_ROOT = os.path.dirname(DATA_ROOT)                   # zip·백업 공통 상위 경로

TEST_IMG = os.path.join(DATA_ROOT, 'test_images')        # 제출용 842장(추론 때 사용)
print("DATA_ROOT:", DATA_ROOT)                           # 채택 경로 확인
print("PROJ_ROOT:", PROJ_ROOT)

In [ ]:
#@title [복원] 기존 56종 원본 5-fold (dataset_5fold.zip)
import os                                                    # 경로 확인 도구

ZIP = os.path.join(PROJ_ROOT, "dataset_5fold.zip")           # 원본 56종 5-fold zip
print("zip 존재:", os.path.exists(ZIP))

!cp "$ZIP" /content/                                          # 드라이브 → 로컬 복사
!unzip -qo /content/dataset_5fold.zip -d /content/dataset      # 압축 해제
print("복원 fold:", sorted(d for d in os.listdir("/content/dataset") if d.startswith("fold")))

In [ ]:
#@title [새 마스킹 기준] 조합단위 COCO 변환 + 5-fold 병합 + zip 백업
import os, json, shutil, pickle                              # 경로·JSON·복사·직렬화 도구
import pandas as pd                                            # 매칭표 로드 도구
from sklearn.model_selection import StratifiedKFold           # 폴드 분할 도구
from PIL import Image                                           # 이미지 크기 확인 도구

TL_DIR = "/content/drive/MyDrive/1팀 공유 문서/김태윤"            # 작업 루트
MASKED = f"{TL_DIR}/masked_combo_56_18"                          # 새 마스킹 결과(조합당 1장)
BASE_5FOLD = "/content/dataset"                                  # 기존 56종 5-fold(원본)
OUT_5FOLD  = "/content/dataset_74"                               # 병합 결과 저장 위치
PROJ_ROOT = "/content/drive/MyDrive/1팀 공유 문서/ai12-level1-project"  # zip 백업 위치

with open(f"{TL_DIR}/combo_boxes_index.pkl", "rb") as f:          # (조합,각도) → 전체 알약 박스
    combo_boxes = pickle.load(f)

DL_18_HAVE = [27652,23222,6562,31704,18109,10220,21025,5093,
              44198,23202,10223,33877,22626,29870,6191]            # 데이터 있는 18종(중 15종)
DL_18_NONE = [139588,140352,152171]                                # 데이터 없는 3종

# [1] 매핑 구성 (기존 56종 id 1~56 유지, 신규 18종 dl_idx 오름차순 57~74)
base0 = json.load(open(f"{BASE_5FOLD}/fold0/train/_annotations.coco.json"))
name_to_id = {c["name"]: c["id"] for c in base0["categories"] if c["id"] != 0}  # K코드(str)→기존id(1~56)
new18 = sorted(DL_18_HAVE + DL_18_NONE)                            # 18종 dl_idx 오름차순
dl18_to_id = {dl: 57 + j for j, dl in enumerate(new18)}            # dl_idx(raw)→57~74

SET_18 = set(DL_18_HAVE)                                            # combo_boxes에 실제 등장 가능한 18종만
TARGET_DLS = SET_18 | {int(c) - 1 for c in name_to_id}              # 전체 타겟(TL 매칭값 기준): 56종은 -1, 18종은 그대로

def cat_of(dl):                                                     # TL 매칭값(dl) → 최종 category_id
    if dl in SET_18:                                                 # 18종이면
        return dl18_to_id[dl]                                        # 57~74 중 하나
    return name_to_id[str(dl + 1)]                                   # 56종이면 dl+1=K코드로 기존id 조회

# [2] 역매핑 저장 (id → 최종 제출값=K코드. 18종은 dl+1로 보정해서 저장)
inv = {v: int(k) for k, v in name_to_id.items()}                     # 56종: id→K코드(이미 정확)
inv.update({v: dl + 1 for dl, v in dl18_to_id.items()})               # 18종: id→dl+1(K코드로 보정)
json.dump(inv, open(f"{TL_DIR}/label_map_74.json", "w"),
          ensure_ascii=False, indent=2)
print(f"매핑 완료: 기존 56 + 신규 {len(new18)} = 74클래스")

# [3] 이미지별 등장 타겟 목록 미리 수집 (파일명 → [(dl, box), ...])
files = sorted([f for f in os.listdir(MASKED) if f.endswith((".png", ".jpg"))])
img_targets = {}                                                     # {파일명: [(dl,box), ...] 타겟만}
for fn in files:
    combo = fn.split("_")[0]                                          # 조합 prefix
    parts = fn.split("_")
    angle = parts[5] if len(parts) > 5 else ""                        # 각도
    boxes = combo_boxes.get((combo, angle), [])                       # 이 이미지 전체 박스
    tgt = [(dl, b) for dl, b in boxes if dl in TARGET_DLS]             # 타겟만 필터
    if tgt:
        img_targets[fn] = tgt

print(f"타겟 포함 이미지 수: {len(img_targets)} / 전체 {len(files)}")

# [4] fold 분할 — 18종 포함 이미지만 5-fold valid 대상, 56종 전용 이미지는 항상 train
val_fold = {}                                                         # {파일명: valid로 갈 fold번호}
labeled_files, labels = [], []                                        # StratifiedKFold 입력용
for fn, tgt in img_targets.items():
    dl18_here = sorted([dl for dl, _ in tgt if dl in SET_18])         # 이 이미지 속 18종들
    if dl18_here:                                                      # 18종이 있으면 fold 분할 대상
        labeled_files.append(fn)
        labels.append(dl18_here[0])                                    # 대표 라벨로 stratify(첫 번째 18종)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)      # seed=42로 5등분
for fold_i, (_, val_idx) in enumerate(skf.split(labeled_files, labels)):
    for vi in val_idx:
        val_fold[labeled_files[vi]] = fold_i                           # 이 fold에서 valid로 갈 파일 지정

# [5] fold별 병합 실행
os.makedirs(OUT_5FOLD, exist_ok=True)
for fold in range(5):
    for split in ["train", "valid"]:
        base = json.load(open(f"{BASE_5FOLD}/fold{fold}/{split}/_annotations.coco.json"))

        exist_ids = {c["id"] for c in base["categories"]}              # 이미 등록된 카테고리
        for dl in new18:                                                # 신규 18종 카테고리 등록(3종 포함)
            nid = dl18_to_id[dl]
            if nid not in exist_ids:
                base["categories"].append({"id": nid, "name": str(dl + 1), "supercategory": "pill"})  # name도 K코드로

        dst = f"{OUT_5FOLD}/fold{fold}/{split}"                         # 저장 경로
        os.makedirs(dst, exist_ok=True)
        nimg = max([i["id"] for i in base["images"]], default=0) + 1    # 다음 이미지 id
        nann = max([a["id"] for a in base["annotations"]], default=0) + 1  # 다음 annotation id

        for fn, tgt in img_targets.items():                             # 새 이미지 하나씩 판단
            dl18_here = [dl for dl, _ in tgt if dl in SET_18]           # 이 이미지 속 18종들
            if dl18_here:                                                # 18종 포함 → fold별 train/valid 분리
                is_val = (val_fold.get(fn) == fold)
                if split == "valid" and not is_val:
                    continue
                if split == "train" and is_val:
                    continue
            else:                                                        # 56종만 있으면 → 항상 train
                if split == "valid":
                    continue

            im = Image.open(os.path.join(MASKED, fn))                    # 이미지 크기 확인
            base["images"].append({"id": nimg, "file_name": fn,
                                    "width": im.width, "height": im.height})
            for dl, box in tgt:                                           # 이 이미지의 타겟 박스 전부 등록 (핵심 변경점)
                base["annotations"].append({
                    "id": nann, "image_id": nimg, "category_id": cat_of(dl),
                    "bbox": box, "area": box[2] * box[3], "iscrowd": 0,
                })
                nann += 1
            shutil.copy(os.path.join(MASKED, fn), os.path.join(dst, fn))  # 이미지 파일 복사
            nimg += 1

        for f in os.listdir(f"{BASE_5FOLD}/fold{fold}/{split}"):          # 기존 원본 이미지 복사
            if f.endswith((".png", ".jpg")):
                shutil.copy(f"{BASE_5FOLD}/fold{fold}/{split}/{f}", os.path.join(dst, f))

        json.dump(base, open(f"{dst}/_annotations.coco.json", "w"), ensure_ascii=False)
        cats_n = len([c for c in base["categories"] if c["id"] != 0])
        print(f"fold{fold}/{split}: img {len(base['images'])} / ann {len(base['annotations'])} / cls {cats_n}")

# [6] 완성 즉시 Drive zip 백업 (기존 파일명 그대로 덮어쓰기)
shutil.make_archive(os.path.join(PROJ_ROOT, "dataset_74_5fold"), "zip", OUT_5FOLD)
print("\n백업 완료:", os.path.join(PROJ_ROOT, "dataset_74_5fold.zip"))